# 04 Limpieza — Normalizar columnas pre-deduplicación

Esta libreta estandariza las columnas bibliográficas restantes antes de deduplicar/fusionar.

Entrada:
`C:\Users\hazar\Desktop\PROYECTO\04_Limpieza\01_Normalizar_Afiliaciones\integrado_solo_unam_normalizado.csv`

Salida:
`C:\Users\hazar\Desktop\PROYECTO\04_Limpieza\02_Normalizar_Columnas\integrado_solo_unam_estandarizado_pre_deduplicacion.csv`

Reglas clave:
- Mantener exactamente las 14 columnas canónicas.
- No deduplicar.
- No normalizar `Autor_norm`, `Afiliacion1`, `Afiliacion2` ni `Area`.
- Dejar `Subarea` siempre vacía.
- Usar celdas vacías reales.
- Corregir `ISBN`, `ISSN`, `Doi`, `URL`, `Titulo`, `Keywords` y `Abstract`.

In [1]:
"""
Fase 04 / 02_Normalizar_Columnas: estandarización bibliográfica pre-deduplicación.

Entrada:
    C:\\Users\\hazar\\Desktop\\PROYECTO\\04_Limpieza\\01_Normalizar_Afiliaciones\\integrado_solo_unam_normalizado.csv

Salida:
    C:\\Users\\hazar\\Desktop\\PROYECTO\\04_Limpieza\\02_Normalizar_Columnas\\integrado_solo_unam_estandarizado_pre_deduplicacion.csv

La libreta/script:
- conserva exactamente las 14 columnas canónicas;
- no deduplica;
- no normaliza Autor_norm, Afiliacion1, Afiliacion2 ni Area;
- deja Subarea siempre vacía;
- usa celdas vacías reales, no N/D, NA, None, etc.;
- corrige DOI, URL, ISBN, ISSN, Titulo, Keywords y Abstract.
"""

from __future__ import annotations

import csv
import html
import json
import re
import unicodedata
from collections import Counter
from pathlib import Path
from typing import Dict, Iterable, List, Tuple

# ---------------------------------------------------------------------
# Rutas del proyecto
# ---------------------------------------------------------------------
RAIZ_PROYECTO = Path(r"C:\Users\hazar\Desktop\PROYECTO")
CARPETA_ENTRADA = RAIZ_PROYECTO / "04_Limpieza" / "01_Normalizar_Afiliaciones"
CARPETA_SALIDA = RAIZ_PROYECTO / "04_Limpieza" / "02_Normalizar_Columnas"

ARCHIVO_ENTRADA = CARPETA_ENTRADA / "integrado_solo_unam_normalizado.csv"
ARCHIVO_SALIDA = CARPETA_SALIDA / "integrado_solo_unam_estandarizado_pre_deduplicacion.csv"

if not ARCHIVO_ENTRADA.exists():
    sandbox_input = Path("/mnt/data/01_Normalizar_Afiliaciones/integrado_solo_unam_normalizado.csv")
    if not sandbox_input.exists():
        sandbox_input = Path("/mnt/data/integrado_solo_unam_normalizado.csv")
    if sandbox_input.exists():
        ARCHIVO_ENTRADA = sandbox_input
        CARPETA_SALIDA = Path("/mnt/data/02_Normalizar_Columnas_v2")
        ARCHIVO_SALIDA = CARPETA_SALIDA / "integrado_solo_unam_estandarizado_pre_deduplicacion.csv"

CARPETA_SALIDA.mkdir(parents=True, exist_ok=True)

CANONICAL_COLUMNS = [
    "indice", "Titulo", "Año", "Autor_norm", "Afiliacion1", "Afiliacion2",
    "ISBN", "ISSN", "Doi", "URL", "Area", "Subarea", "Keywords", "Abstract",
]

EMPTY_EQUIVALENTS = {
    "", "n/d", "nd", "n.a", "n/a", "na", "nan", "none", "null", "-", "--", "—",
    "sin informacion", "sin información", "sin dato", "sin datos", "no disponible",
    "not available", "no information", "s/i", "s d", "sd",
}

MONTHS = {
    "jan": 1, "january": 1, "ene": 1, "enero": 1,
    "feb": 2, "february": 2, "febrero": 2,
    "mar": 3, "march": 3, "marzo": 3,
    "apr": 4, "april": 4, "abr": 4, "abril": 4,
    "may": 5, "mayo": 5,
    "jun": 6, "june": 6, "junio": 6,
    "jul": 7, "july": 7, "julio": 7,
    "aug": 8, "august": 8, "ago": 8, "agosto": 8,
    "sep": 9, "sept": 9, "september": 9, "septiembre": 9,
    "oct": 10, "october": 10, "octubre": 10,
    "nov": 11, "november": 11, "noviembre": 11,
    "dec": 12, "december": 12, "dic": 12, "diciembre": 12,
}

ACRONYMS = {
    "ACM", "AI", "API", "AR", "BERT", "CNN", "CPU", "CT", "DNA", "DNN", "DOI", "DRL",
    "GPU", "GNN", "HCI", "HTML", "IBDATA", "IEEE", "IoT", "IP", "LLM", "LLMS", "LSTM",
    "ML", "MRI", "NLP", "ODE", "PCA", "RAG", "RF", "RNA", "RNN", "SVM", "UNAM", "URL",
    "VR", "XML", "XR", "T2", "7T", "ALMA", "COVID", "COVID-19", "NMR", "PET", "SEM",
    "TEM", "PINN", "GAN", "NLP", "NLI", "UAV", "RGB", "FEM", "GPU", "CPU", "PDF",
}

TITLE_SPECIAL_CASE = {
    "NEURALODE": "NeuralODE",
    "DEEPONET": "DeepONet",
    "PHYSICSINFORMED": "Physics-Informed",
}

TITLE_SMALL_WORDS = {
    "a", "an", "and", "as", "at", "by", "de", "del", "for", "from", "in", "la", "las",
    "of", "on", "or", "para", "por", "the", "to", "with", "y",
}

DOI_RE = re.compile(r"10\.\d{4,9}/[^\s\"'<>]+", flags=re.I)
DOI_PREFIX_RE = re.compile(r"^(?:https?://(?:dx\.)?doi\.org/|(?:dx\.)?doi\.org/|doi\s*[:/]\s*|doi/)", flags=re.I)
ISSN_RE = re.compile(r"\b(\d{4})[-\s]?(\d{3}[0-9Xx])\b")
SCIENTIFIC_RE = re.compile(r"^[+-]?\d+(?:\.\d+)?[eE][+-]?\d+$")
HTML_TAG_RE = re.compile(r"</?\s*[A-Za-z][A-Za-z0-9]*(?:\s+[^<>]*)?>")
URL_RE = re.compile(r"https?://\S+|www\.\S+", flags=re.I)

# ---------------------------------------------------------------------
# Utilidades generales
# ---------------------------------------------------------------------
def strip_accents(text: object) -> str:
    text = "" if text is None else str(text)
    return "".join(
        char for char in unicodedata.normalize("NFKD", text)
        if not unicodedata.combining(char)
    )


def norm_for_empty(text: object) -> str:
    text = strip_accents(text).lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text


def norm_soft(text: object) -> str:
    text = strip_accents(text).lower()
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def clean_spaces(text: object) -> str:
    text = "" if text is None else str(text)
    text = html.unescape(text)
    text = text.replace("\u00a0", " ")
    text = text.replace("\r", " ").replace("\n", " ").replace("\t", " ")
    return re.sub(r"\s+", " ", text).strip()


def empty_if_placeholder(text: object) -> str:
    value = clean_spaces(text)
    if norm_for_empty(value) in EMPTY_EQUIVALENTS:
        return ""
    return value


def remove_html(text: str, replace_breaks_with: str = " ") -> str:
    text = html.unescape(str(text or ""))
    text = re.sub(r"<\s*br\s*/?\s*>", replace_breaks_with, text, flags=re.I)
    text = re.sub(r"<\s*/\s*p\s*>", replace_breaks_with, text, flags=re.I)
    text = HTML_TAG_RE.sub(" ", text)
    return clean_spaces(text)


def unique_preserve_order(items: Iterable[str], key_func=None) -> List[str]:
    out: List[str] = []
    seen = set()
    for item in items:
        item = clean_spaces(item)
        if not item:
            continue
        key = key_func(item) if key_func else item.lower()
        if key not in seen:
            seen.add(key)
            out.append(item)
    return out

# ---------------------------------------------------------------------
# indice
# ---------------------------------------------------------------------
def normalize_indice(value: object) -> Tuple[str, List[str]]:
    flags: List[str] = []
    raw = empty_if_placeholder(value)
    if not raw:
        return "", ["indice_vacio"]
    raw_no_commas = raw.replace(",", "")
    if re.fullmatch(r"\d+", raw_no_commas):
        return str(int(raw_no_commas)), flags
    m = re.fullmatch(r"(\d+)\.0+", raw_no_commas)
    if m:
        flags.append("indice_decimal_convertido_a_entero")
        return str(int(m.group(1))), flags
    flags.append("indice_no_entero_revisar")
    return raw, flags

# ---------------------------------------------------------------------
# Titulo
# ---------------------------------------------------------------------
def uppercase_ratio(title: str) -> float:
    letters = [ch for ch in title if ch.isalpha()]
    if not letters:
        return 0.0
    return sum(1 for ch in letters if ch.isupper()) / len(letters)


def should_convert_title_caps(title: str) -> bool:
    letters = [ch for ch in title if ch.isalpha()]
    return bool(letters) and uppercase_ratio(title) >= 0.88 and not any(ch.islower() for ch in letters[:max(1, int(len(letters) * 0.1))])


def smart_title_word(word: str, position: int) -> str:
    prefix = re.match(r"^\W*", word).group(0)
    suffix = re.search(r"\W*$", word).group(0)
    core = word[len(prefix): len(word) - len(suffix) if suffix else len(word)]
    if not core:
        return word
    clean = re.sub(r"[^A-Za-z0-9ÁÉÍÓÚÜÑáéíóúüñ-]", "", core)
    clean_no_acc_upper = strip_accents(clean).upper()
    clean_no_acc_lower = strip_accents(clean).lower()
    if clean_no_acc_upper in TITLE_SPECIAL_CASE:
        return prefix + TITLE_SPECIAL_CASE[clean_no_acc_upper] + suffix
    if clean_no_acc_upper in ACRONYMS or any(ch.isdigit() for ch in clean):
        return prefix + core.upper() + suffix
    if position > 0 and clean_no_acc_lower in TITLE_SMALL_WORDS:
        return prefix + clean_no_acc_lower + suffix

    def cap_segment(seg: str) -> str:
        if not seg:
            return seg
        # Conserva segmentos que son acrónimos conocidos.
        if strip_accents(seg).upper() in ACRONYMS:
            return seg.upper()
        return seg[0].upper() + seg[1:].lower()

    return prefix + "-".join(cap_segment(seg) for seg in core.split("-")) + suffix


def normalize_title(value: object) -> Tuple[str, List[str]]:
    flags: List[str] = []
    title = empty_if_placeholder(value)
    title = re.sub(r"\s+", " ", title).strip()
    if not title:
        return "", ["titulo_vacio"]

    title2 = re.sub(r"\s+(?:https?://\S+|doi\s*:?\s*10\.\S+)$", "", title, flags=re.I).strip()
    if title2 != title:
        flags.append("titulo_metadato_doi_url_removido")
        title = title2

    if should_convert_title_caps(title):
        tokens = re.split(r"(\s+)", title)
        position = 0
        rebuilt = []
        for token in tokens:
            if token.isspace():
                rebuilt.append(token)
            else:
                rebuilt.append(smart_title_word(token, position))
                position += 1
        new_title = "".join(rebuilt).strip()
        if new_title != title:
            flags.append("titulo_mayusculas_convertido_capitalizacion_normal")
            title = new_title

    # Corrección explícita por si el título ya fue mal capitalizado en una ejecución previa.
    title = re.sub(r"\bNeuralode\b", "NeuralODE", title)
    return title, flags

# ---------------------------------------------------------------------
# Año
# ---------------------------------------------------------------------
def detect_month(value: str) -> int | None:
    value_norm = strip_accents(value).lower()
    for name, month in MONTHS.items():
        if re.search(rf"\b{re.escape(name)}\b", value_norm):
            return month
    m_iso = re.search(r"\b(20\d{2})[-/](\d{1,2})(?:[-/]\d{1,2})?\b", value_norm)
    if m_iso:
        month = int(m_iso.group(2))
        if 1 <= month <= 12:
            return month
    m_dmy = re.search(r"\b\d{1,2}[-/](\d{1,2})[-/](20\d{2})\b", value_norm)
    if m_dmy:
        month = int(m_dmy.group(1))
        if 1 <= month <= 12:
            return month
    return None


def has_full_or_month_date(value: str) -> bool:
    v = strip_accents(value).lower()
    if any(re.search(rf"\b{re.escape(name)}\b", v) for name in MONTHS):
        return True
    if re.search(r"\b20\d{2}[-/]\d{1,2}(?:[-/]\d{1,2})?\b", v):
        return True
    if re.search(r"\b\d{1,2}[-/]\d{1,2}[-/]20\d{2}\b", v):
        return True
    return False


def normalize_year(value: object) -> Tuple[str, List[str]]:
    flags: List[str] = []
    raw = empty_if_placeholder(value)
    if not raw:
        return "", ["anio_vacio"]
    m = re.search(r"\b(20\d{2})\b", raw)
    if not m:
        return "", ["anio_no_detectado"]
    year = int(m.group(1))
    if year not in {2024, 2025}:
        return "", [f"anio_fuera_proyecto_{year}"]
    month = detect_month(raw)
    if has_full_or_month_date(raw) and month is not None:
        if year == 2024 and month < 3:
            flags.append("fecha_completa_fuera_rango_antes_marzo_2024_revisar")
    return str(year), flags

# ---------------------------------------------------------------------
# ISBN
# ---------------------------------------------------------------------
def isbn10_valid(chars: str) -> bool:
    chars = chars.upper()
    if not re.fullmatch(r"\d{9}[\dX]", chars):
        return False
    total = sum((10 - i) * (10 if ch == "X" else int(ch)) for i, ch in enumerate(chars))
    return total % 11 == 0


def isbn13_valid(chars: str) -> bool:
    if not re.fullmatch(r"\d{13}", chars):
        return False
    total = sum((1 if i % 2 == 0 else 3) * int(ch) for i, ch in enumerate(chars[:12]))
    check = (10 - (total % 10)) % 10
    return check == int(chars[-1])


def hyphenate_isbn13(chars: str) -> str:
    """Guionización práctica para los patrones presentes en la base.

    No pretende sustituir el registro oficial de ISBN, pero evita salidas compactas
    o guiones incompletos. Sólo se aplica después de validar checksum.
    """
    prefix, rest, check = chars[:3], chars[3:-1], chars[-1]

    if chars.startswith("978303") or chars.startswith("978331"):
        return f"{prefix}-3-{chars[4:7]}-{chars[7:12]}-{check}"
    if chars.startswith("978981"):
        return f"{prefix}-981-{chars[6:8]}-{chars[8:12]}-{check}"
    if chars.startswith("978607"):
        return f"{prefix}-607-{chars[6:8]}-{chars[8:12]}-{check}"
    if chars.startswith("97884"):
        return f"{prefix}-84-{chars[5:9]}-{chars[9:12]}-{check}"
    if chars.startswith("9780") or chars.startswith("9781"):
        return f"{prefix}-{chars[3]}-{chars[4:8]}-{chars[8:12]}-{check}"
    if chars.startswith("9798"):
        return f"{prefix}-8-{chars[4:8]}-{chars[8:12]}-{check}"

    # Fallback conservador por grupos aparentes.
    if rest.startswith(("0", "1")):
        return f"{prefix}-{rest[0]}-{rest[1:5]}-{rest[5:]}-{check}"
    if rest.startswith("3"):
        return f"{prefix}-3-{rest[1:4]}-{rest[4:]}-{check}"
    if rest.startswith(("607", "607")):
        return f"{prefix}-{rest[:3]}-{rest[3:5]}-{rest[5:]}-{check}"
    return f"{prefix}-{rest}-{check}"


def hyphenate_isbn10(chars: str) -> str:
    chars = chars.upper()
    check = chars[-1]
    if chars[0] in {"0", "1"}:
        return f"{chars[0]}-{chars[1:5]}-{chars[5:9]}-{check}"
    if chars[0] in {"2", "3"}:
        return f"{chars[0]}-{chars[1:5]}-{chars[5:9]}-{check}"
    return f"{chars[:-1]}-{check}"


def split_identifier_candidates(value: str) -> List[str]:
    value = empty_if_placeholder(value)
    if not value:
        return []
    value = re.sub(r"\b(?:e?isbn(?:-1[03])?|issn|eissn|e-issn)\s*[:#]?", "", value, flags=re.I)
    value = value.replace("\n", ",").replace("\r", ",")
    value = re.sub(r"\s*[;|]\s*", ",", value)
    value = re.sub(r"\s*,\s*", ",", value)
    return [p.strip() for p in value.split(",") if p.strip()]


def normalize_isbn(value: object) -> Tuple[str, List[str]]:
    flags: List[str] = []
    raw = empty_if_placeholder(value)
    if not raw:
        return "", flags
    out: List[str] = []
    invalid: List[str] = []
    for cand in split_identifier_candidates(raw):
        c = clean_spaces(cand).strip(" .,:;")
        if SCIENTIFIC_RE.fullmatch(c):
            flags.append("isbn_notacion_cientifica_eliminado_revisar_pdf")
            continue
        c = re.sub(r"\.0+$", "", c)
        chars = re.sub(r"[^0-9Xx]", "", c).upper()
        if len(chars) == 13 and isbn13_valid(chars):
            out.append(hyphenate_isbn13(chars))
        elif len(chars) == 10 and isbn10_valid(chars):
            out.append(hyphenate_isbn10(chars))
        elif len(chars) in {10, 13}:
            invalid.append(c)
        else:
            invalid.append(c)
    if invalid:
        flags.append("isbn_invalidos_eliminados_revisar_pdf")
    out = unique_preserve_order(out, key_func=lambda x: re.sub(r"[^0-9Xx]", "", x).upper())
    return ", ".join(out), flags

# ---------------------------------------------------------------------
# ISSN
# ---------------------------------------------------------------------
def issn_valid(chars: str) -> bool:
    chars = chars.upper().replace("-", "")
    if not re.fullmatch(r"\d{7}[\dX]", chars):
        return False
    total = 0
    for i, ch in enumerate(chars):
        value = 10 if ch == "X" else int(ch)
        total += (8 - i) * value
    return total % 11 == 0


def normalize_issn(value: object) -> Tuple[str, List[str]]:
    flags: List[str] = []
    raw = empty_if_placeholder(value)
    if not raw:
        return "", flags
    raw_clean = re.sub(r"\b(?:eissn|e-issn|issn)\s*[:#]?", "", raw, flags=re.I)
    found = []
    for m in ISSN_RE.finditer(raw_clean):
        token = f"{m.group(1)}-{m.group(2).upper()}"
        if issn_valid(token):
            found.append(token)
        else:
            flags.append("issn_checksum_invalido_eliminado_revisar")
    found = unique_preserve_order(found, key_func=lambda x: x.upper())
    if not found and raw_clean.strip():
        flags.append("issn_invalido_eliminado_revisar")
    return ", ".join(found), flags

# ---------------------------------------------------------------------
# DOI / URL
# ---------------------------------------------------------------------
def extract_dois(*values: str) -> List[str]:
    dois: List[str] = []
    for value in values:
        if not value:
            continue
        text = html.unescape(str(value)).strip()
        text = DOI_PREFIX_RE.sub("", text)
        for m in DOI_RE.finditer(text):
            doi = m.group(0).strip()
            doi = DOI_PREFIX_RE.sub("", doi)
            doi = doi.rstrip(".,;:)]}>")
            doi = doi.lower()
            if doi.startswith("10."):
                dois.append(doi)
    return unique_preserve_order(dois, key_func=lambda x: x.lower())


def normalize_doi(value: object, url_value: object = "") -> Tuple[str, List[str]]:
    flags: List[str] = []
    raw = empty_if_placeholder(value)
    url = empty_if_placeholder(url_value)
    dois = extract_dois(raw)
    if not dois and url:
        from_url = extract_dois(url)
        if from_url:
            dois = from_url
            flags.append("doi_recuperado_desde_url")
    if not raw and not dois:
        return "", flags
    if len(dois) > 1:
        flags.append("doi_multiple_se_conservo_primero_revisar")
    if dois:
        return dois[0], flags
    if raw:
        flags.append("doi_invalido_eliminado_revisar")
    return "", flags


def normalize_url(value: object, doi: str) -> Tuple[str, List[str]]:
    flags: List[str] = []
    raw = empty_if_placeholder(value)
    if raw:
        raw = raw.strip()
        if re.match(r"^(?:https?://|www\.|doi\.org/|10\.)", raw, flags=re.I):
            raw = re.sub(r"\s+", "", raw)
        else:
            raw = clean_spaces(raw)
        if raw.lower().startswith("doi.org/"):
            return (f"https://doi.org/{doi}" if doi else "https://" + raw), ["url_doi_normalizada"]
        if raw.lower().startswith("www."):
            return "https://" + raw, ["url_www_https_agregado"]
        if raw.lower().startswith("10."):
            return f"https://doi.org/{doi or raw.lower()}", ["url_doi_normalizada"]
        if re.match(r"^https?://", raw, flags=re.I):
            if "doi.org/" in raw.lower() and doi:
                return f"https://doi.org/{doi}", ["url_doi_normalizada"]
            return raw, flags
        flags.append("url_invalida_eliminada_revisar")
        return "", flags
    if doi:
        return f"https://doi.org/{doi}", ["url_creada_desde_doi"]
    return "", flags

# ---------------------------------------------------------------------
# Keywords
# ---------------------------------------------------------------------
def keyword_soft_key(token: str) -> str:
    # Deduplicación suave: acentos fuera, minúsculas, guiones y puntuación como espacios.
    return norm_soft(token)


def normalize_keyword_token(token: str) -> str:
    token = remove_html(token, replace_breaks_with=" ")
    token = URL_RE.sub(" ", token)
    token = re.sub(r"\b(?:keywords?|author keywords?|index terms?|ccs concepts?)\b\s*[:\-]", "", token, flags=re.I)
    token = re.sub(r"\s+", " ", token).strip(" ,;:.-•\u2022")
    token = token.lower()
    token = re.sub(r"\s+", " ", token).strip()
    plain = strip_accents(token).upper()
    if plain in ACRONYMS:
        return plain
    return token


def normalize_keywords(value: object) -> Tuple[str, List[str]]:
    flags: List[str] = []
    raw = empty_if_placeholder(value)
    if not raw:
        return "", flags
    text = html.unescape(raw)
    if HTML_TAG_RE.search(text):
        flags.append("keywords_html_removido")
    if URL_RE.search(text):
        flags.append("keywords_url_removida")
    text = re.sub(r"<\s*br\s*/?\s*>", ",", text, flags=re.I)
    text = HTML_TAG_RE.sub(" ", text)
    text = URL_RE.sub(" ", text)
    text = re.sub(r"\b(?:keywords?|author keywords?|index terms?|ccs concepts?)\b\s*[:\-]", "", text, flags=re.I)
    text = text.replace("\r", ",").replace("\n", ",").replace("\t", ",")
    text = re.sub(r"\s*[;|•\u2022]\s*", ",", text)
    # Separadores de lista con guion largo o guion rodeado de espacios. No toca guiones internos.
    text = re.sub(r"\s+[–—-]\s+", ",", text)
    text = re.sub(r",{2,}", ",", text)
    pieces = [normalize_keyword_token(p) for p in text.split(",")]
    pieces = [p for p in pieces if p and norm_for_empty(p) not in EMPTY_EQUIVALENTS]
    before = len(pieces)
    pieces = unique_preserve_order(pieces, key_func=keyword_soft_key)
    if len(pieces) < before:
        flags.append("keywords_duplicadas_eliminadas_clave_suave")
    return ", ".join(pieces), flags

# ---------------------------------------------------------------------
# Abstract
# ---------------------------------------------------------------------
def remove_latex_preamble(text: str) -> Tuple[str, bool]:
    original = text
    text = re.sub(r"\\documentclass.*?(?:\\begin\{document\}|$)", " ", text, flags=re.I | re.S)
    text = re.sub(r"\\usepackage(?:\[[^\]]*\])?\{[^}]+\}", " ", text, flags=re.I)
    text = re.sub(r"\\setlength\{[^}]+\}\{[^}]+\}", " ", text, flags=re.I)
    text = re.sub(r"\\(?:begin|end)\{document\}", " ", text, flags=re.I)
    text = re.sub(r"\bdocumentclass\b.*?(?:\bbegin document\b|\bend document\b|$)", " ", text, flags=re.I | re.S)
    text = re.sub(r"\busepackage\b\s*(?:\[[^\]]*\])?\s*\w+", " ", text, flags=re.I)
    text = re.sub(r"\bsetlength\b\s*\w+\s*\w+", " ", text, flags=re.I)
    return clean_spaces(text), text != original


def replace_latex_frac(text: str) -> str:
    # Repite para fracciones simples no anidadas.
    pattern = re.compile(r"\\frac\s*\{([^{}]+)\}\s*\{([^{}]+)\}")
    previous = None
    while previous != text:
        previous = text
        text = pattern.sub(lambda m: f"{m.group(1).strip()}/{m.group(2).strip()}", text)
    return text


def clean_latex_symbols(text: str) -> Tuple[str, bool]:
    original = text
    text, preamble_changed = remove_latex_preamble(text)
    text = replace_latex_frac(text)
    text = re.sub(r"\\(?:mathbf|mathsf|mathrm|textrm|mathit|operatorname|text)\s*\{([^{}]*)\}", r"\1", text, flags=re.I)
    text = re.sub(r"\\(?:left|right|displaystyle|!|,|;|:|quad|qquad)\b", " ", text, flags=re.I)
    greek = {
        "alpha": "alpha", "beta": "beta", "gamma": "gamma", "delta": "delta", "epsilon": "epsilon",
        "varepsilon": "epsilon", "theta": "theta", "vartheta": "theta", "lambda": "lambda", "mu": "mu",
        "pi": "pi", "rho": "rho", "sigma": "sigma", "tau": "tau", "phi": "phi", "varphi": "phi",
        "omega": "omega", "Omega": "Omega", "Theta": "Theta",
    }
    for cmd, repl in greek.items():
        text = re.sub(rf"\\{cmd}\b", repl, text)
    text = re.sub(r"\\(?:infty|leq?|geq?|neq|approx|times|cdot|sum|prod|sqrt)\b", " ", text, flags=re.I)
    text = re.sub(r"\\[A-Za-z]+", " ", text)  # comandos restantes no interpretables.
    text = re.sub(r"\\\(|\\\)|\\\[|\\\]", " ", text)
    text = text.replace("$", " ")
    text = text.replace("<^>", "^")
    text = re.sub(r"&\s*(?:varepsilon|epsilon)\s*;?", "epsilon", text, flags=re.I)
    text = re.sub(r"\bvarepsilon\b", "epsilon", text, flags=re.I)
    text = re.sub(r"\b(?:mathsf|mathbf|mathrm|textrm|mathit|operatorname|displaystyle|frac)\b", " ", text, flags=re.I)
    text = re.sub(r"\b(?:left|right)\s*([()\[\]])", r"\1", text, flags=re.I)
    text = re.sub(r"[{}]", " ", text)
    text = re.sub(r"\s+([,.;:!?])", r"\1", text)
    text = re.sub(r"([([{])\s+", r"\1", text)
    text = re.sub(r"\s+([)\]}])", r"\1", text)
    text = clean_spaces(text)
    return text, (text != original or preamble_changed)


def remove_legal_blocks(text: str) -> Tuple[str, bool]:
    original = text
    # Bloques legales frecuentes en bases bibliográficas.
    text = re.sub(
        r"\[?\s*The copyright for the referenced work is owned by author\(s\).*?authorized sources\.?\s*\]?",
        " ", text, flags=re.I | re.S,
    )
    text = re.sub(r"\bCopies of full-text articles.*?authorized sources\.?", " ", text, flags=re.I | re.S)

    # Eliminar colas legales. Importante: NO elimina '(c)' si no va seguido de año/legal.
    tail_patterns = [
        r"\s*©.*$",
        r"\s*(?:\(c\))\s*(?:copyright\s*)?(?:author\s*\(?s\)?\s*)?\d{4}.*$",
        r"\s*copyright\s*(?:author\s*\(?s\)?\s*)?\d{4}.*$",
        r"\s*copyright\s*$",
        r"\s*all rights reserved.*$",
        r"\s*this (?:article|work) is (?:an )?open access.*$",
        r"\s*published by .*?$",
        r"\s*under (?:exclusive )?licen[cs]e .*?$",
        r"\s*licensed under .*?$",
    ]
    for pat in tail_patterns:
        text = re.sub(pat, "", text, flags=re.I | re.S).strip()
    return clean_spaces(text), text != original


def remove_availability_sentences(text: str) -> Tuple[str, bool]:
    """Quita frases finales tipo 'Code is available at ...'.

    No elimina cualquier URL del abstract, sólo frases de disponibilidad al final.
    """
    original = text
    sentences = re.split(r"(?<=[.!?])\s+", text)
    if len(sentences) <= 1:
        return text, False
    availability_re = re.compile(
        r"(?i)(code is available|our code|source code|project page|database url|data are available|"
        r"dataset is available|available at|freely available|implementation is available|software is available)"
    )
    changed = False
    while sentences and (URL_RE.search(sentences[-1]) and availability_re.search(sentences[-1])):
        sentences.pop()
        changed = True
    text = " ".join(sentences).strip()
    return clean_spaces(text), changed or text != original


def abstract_is_only_legal(text: str) -> bool:
    n = norm_soft(text)
    if not n:
        return True
    legal_terms = [
        "copyright", "all rights reserved", "published by", "creative commons", "open access",
        "license", "licence", "author s", "authors", "elsevier", "springer", "ieee", "acm", "mdpi",
    ]
    if len(n.split()) <= 18 and any(term in n for term in legal_terms):
        return True
    return False


def remove_author_list_from_edges(text: str, author: str) -> Tuple[str, bool]:
    original = text
    author = clean_spaces(author)
    if not author:
        return text, False
    variants = [re.escape(author)]
    if "," in author:
        last, given = [p.strip() for p in author.split(",", 1)]
        if last and given:
            variants.append(re.escape(f"{given} {last}"))
    pat = r"(?:" + "|".join(variants) + r")"
    text = re.sub(rf"^\s*{pat}\s*[:;,\.\-–—]?\s*", "", text, flags=re.I)
    text = re.sub(rf"\s*[:;,\.\-–—]?\s*{pat}\s*$", "", text, flags=re.I)
    return clean_spaces(text), text != original


def normalize_abstract(value: object, author: object = "") -> Tuple[str, List[str]]:
    flags: List[str] = []
    raw = empty_if_placeholder(value)
    if not raw:
        return "", flags
    text = html.unescape(raw)
    if HTML_TAG_RE.search(text):
        flags.append("abstract_html_removido")
    text = remove_html(text, replace_breaks_with=" ")
    text = re.sub(r"^\s*(?:abstract|resumen)\s*[:.\-–—]?\s*", "", text, flags=re.I).strip()

    text, changed_legal = remove_legal_blocks(text)
    if changed_legal:
        flags.append("abstract_copyright_legal_removido")

    text, changed_avail = remove_availability_sentences(text)
    if changed_avail:
        flags.append("abstract_frase_disponibilidad_url_removida")

    # Después de quitar frases de disponibilidad, eliminar URLs residuales del abstract.
    if URL_RE.search(text):
        text = URL_RE.sub("", text)
        flags.append("abstract_url_residual_removida")
        text = clean_spaces(text)

    text, changed_latex = clean_latex_symbols(text)
    if changed_latex:
        flags.append("abstract_latex_limpiado")

    text, changed_author = remove_author_list_from_edges(text, str(author or ""))
    if changed_author:
        flags.append("abstract_nombre_autor_en_borde_removido")

    # Limpieza final de residuos muy específicos.
    text = re.sub(r"\b(?:authorized sources|full-text articles)\b.*$", "", text, flags=re.I).strip()
    text = re.sub(r"\s+([,.;:!?])", r"\1", text)
    text = clean_spaces(text)

    if abstract_is_only_legal(text):
        flags.append("abstract_solo_copyright_licencia_vaciado")
        return "", flags
    return text, flags

# ---------------------------------------------------------------------
# Validación breve
# ---------------------------------------------------------------------
def validate_final_row(row: Dict[str, str]) -> List[str]:
    issues: List[str] = []
    if not re.fullmatch(r"\d+", row.get("indice", "")):
        issues.append("indice_no_entero")
    if row.get("Año", "") not in {"2024", "2025"}:
        issues.append("anio_fuera_2024_2025")
    if row.get("Subarea", "") != "":
        issues.append("subarea_no_vacia")
    if row.get("Doi", "") and not row["Doi"].startswith("10."):
        issues.append("doi_no_inicia_10")
    if row.get("URL", "") and not re.match(r"^https?://", row["URL"], flags=re.I):
        issues.append("url_no_http")
    kw = row.get("Keywords", "")
    if kw and (";" in kw or "|" in kw or HTML_TAG_RE.search(kw) or URL_RE.search(kw)):
        issues.append("keywords_separador_html_o_url")
    abstract = row.get("Abstract", "")
    if abstract:
        if HTML_TAG_RE.search(abstract):
            issues.append("abstract_html")
        if re.search(r"(?i)(copyright|all rights reserved|authorized sources|copies of full-text articles)", abstract):
            issues.append("abstract_legal_residual")
        if re.search(r"(?i)\b(?:documentclass|usepackage|setlength|mathsf|mathbf|mathrm|textrm|operatorname|displaystyle|frac)\b", abstract):
            issues.append("abstract_latex_residual")
    return issues

# ---------------------------------------------------------------------
# Proceso principal
# ---------------------------------------------------------------------
def read_rows(path: Path) -> List[Dict[str, str]]:
    with path.open("r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        if reader.fieldnames != CANONICAL_COLUMNS:
            raise ValueError(f"Columnas no canónicas. Esperadas={CANONICAL_COLUMNS}; recibidas={reader.fieldnames}")
        return [{k: (v if v is not None else "") for k, v in row.items()} for row in reader]


def write_rows(path: Path, rows: List[Dict[str, str]]) -> None:
    with path.open("w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=CANONICAL_COLUMNS, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)


def process_rows(rows: List[Dict[str, str]]) -> Tuple[List[Dict[str, str]], Dict[str, object]]:
    final_rows: List[Dict[str, str]] = []
    flags_counter = Counter()
    validation_counter = Counter()
    changed_counter = Counter()

    for row in rows:
        out = {col: "" for col in CANONICAL_COLUMNS}
        flags_by_col: Dict[str, List[str]] = {col: [] for col in CANONICAL_COLUMNS}

        out["indice"], flags_by_col["indice"] = normalize_indice(row.get("indice", ""))
        out["Titulo"], flags_by_col["Titulo"] = normalize_title(row.get("Titulo", ""))
        out["Año"], flags_by_col["Año"] = normalize_year(row.get("Año", ""))

        # En esta fase NO se normalizan Autor_norm, Afiliacion1, Afiliacion2 ni Area.
        out["Autor_norm"] = empty_if_placeholder(row.get("Autor_norm", ""))
        out["Afiliacion1"] = empty_if_placeholder(row.get("Afiliacion1", ""))
        out["Afiliacion2"] = empty_if_placeholder(row.get("Afiliacion2", ""))
        out["Area"] = empty_if_placeholder(row.get("Area", ""))

        out["ISBN"], flags_by_col["ISBN"] = normalize_isbn(row.get("ISBN", ""))
        out["ISSN"], flags_by_col["ISSN"] = normalize_issn(row.get("ISSN", ""))
        out["Doi"], flags_by_col["Doi"] = normalize_doi(row.get("Doi", ""), row.get("URL", ""))
        out["URL"], flags_by_col["URL"] = normalize_url(row.get("URL", ""), out["Doi"])
        out["Keywords"], flags_by_col["Keywords"] = normalize_keywords(row.get("Keywords", ""))
        out["Abstract"], flags_by_col["Abstract"] = normalize_abstract(row.get("Abstract", ""), row.get("Autor_norm", ""))

        out["Subarea"] = ""
        if empty_if_placeholder(row.get("Subarea", "")):
            flags_by_col["Subarea"].append("subarea_vaciada_por_regla")

        # Vacíos reales en todas las columnas.
        for col in CANONICAL_COLUMNS:
            out[col] = empty_if_placeholder(out.get(col, ""))

        final_rows.append(out)

        for col in CANONICAL_COLUMNS:
            if (row.get(col, "") or "") != out.get(col, ""):
                changed_counter[col] += 1
            for flag in flags_by_col.get(col, []):
                flags_counter[f"{col}:{flag}"] += 1
        for issue in validate_final_row(out):
            validation_counter[issue] += 1

    summary = {
        "archivo_entrada": str(ARCHIVO_ENTRADA),
        "archivo_salida": str(ARCHIVO_SALIDA),
        "filas": len(final_rows),
        "columnas": len(CANONICAL_COLUMNS),
        "columnas_canonicas_ok": True,
        "cambios_por_columna": dict(changed_counter),
        "flags_limpieza": dict(flags_counter),
        "issues_validacion": dict(validation_counter),
    }
    return final_rows, summary


def main() -> None:
    rows = read_rows(ARCHIVO_ENTRADA)
    final_rows, summary = process_rows(rows)
    write_rows(ARCHIVO_SALIDA, final_rows)

    print("Normalización pre-deduplicación terminada.")
    print(json.dumps(summary, ensure_ascii=False, indent=2))
    print(f"Archivo final: {ARCHIVO_SALIDA}")


if __name__ == "__main__":
    main()


Normalización pre-deduplicación terminada.
{
  "archivo_entrada": "C:\\Users\\hazar\\Desktop\\PROYECTO\\04_Limpieza\\01_Normalizar_Afiliaciones\\integrado_solo_unam_normalizado.csv",
  "archivo_salida": "C:\\Users\\hazar\\Desktop\\PROYECTO\\04_Limpieza\\02_Normalizar_Columnas\\integrado_solo_unam_estandarizado_pre_deduplicacion.csv",
  "filas": 3239,
  "columnas": 14,
  "columnas_canonicas_ok": true,
  "cambios_por_columna": {
    "Keywords": 3141,
    "Abstract": 1582,
    "URL": 1114,
    "ISBN": 817,
    "Titulo": 14
  },
  "flags_limpieza": {
    "URL:url_doi_normalizada": 502,
    "Abstract:abstract_latex_limpiado": 124,
    "Abstract:abstract_html_removido": 4,
    "URL:url_creada_desde_doi": 763,
    "Abstract:abstract_copyright_legal_removido": 1436,
    "Keywords:keywords_duplicadas_eliminadas_clave_suave": 108,
    "Titulo:titulo_mayusculas_convertido_capitalizacion_normal": 14,
    "Abstract:abstract_frase_disponibilidad_url_removida": 32,
    "Keywords:keywords_html_removid